<a href="https://colab.research.google.com/github/carrilloluis25/ECE_551_problems_ML/blob/main/notebooks/fast_ddpm_parallel_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fast-DDPM inference on several GPUs

[`fast_ddpm_viewer.ipynb`](fast_ddpm_viewer.ipynb) runs the model on a few images. This notebook extends that example to the questions the HPC project asks: **how fast is inference, and what changes when more than one GPU is used?**

| Section | What it shows | Needs |
|---|---|---|
| 1 | One GPU: images per second at different batch sizes, and what 10, 5 and 2 sampling steps cost in time and quality | 1 GPU |
| 2 | **Splitting one batch across GPUs** by hand: what `torch.nn.DataParallel` does inside the training and inference code | 1 GPU (runs the pieces one after another) or 2+ GPUs (runs them at the same time) |
| 3 | **Sharding the test set**: each GPU takes every N-th image and works alone, then the results are merged. This is how the cluster will run inference (a job array, or one `torchrun` launch). The check: the merged result must equal the one-GPU result. | 1 GPU or more |
| 4 | Look at the images | |
| 5 | Plot the timings measured on Easley (upload the `scale_result_*.txt` files from `slurm/scaling_bench.sh`) | no GPU |

**Be clear about what Colab can show.** A free Colab runtime has one GPU, so with one GPU sections 2 and 3 run the pieces one after another. That proves the splitting and merging are *correct*, but the wall-clock time does not get shorter. The real multi-GPU timings come from Easley and are plotted in section 5.

**Words used here**
- **batch**: images sent through the model together in one call.
- **device / GPU**: one graphics card, named `cuda:0`, `cuda:1`, ...
- **replica**: a copy of the model's weights on one device.
- **shard**: a subset of the test set. With N shards, shard *k* holds the images whose position in the list leaves remainder *k* when divided by N.
- **worker**: one GPU working on one shard or one chunk of a batch.
- **wall-clock time**: seconds on a clock, from start to finish of the whole job.
- **throughput**: images produced per second.
- **speed-up**: (time with 1 GPU) / (time with N GPUs). **Efficiency**: speed-up / N, where 1.0 is perfect.
- **sampling steps**: how many times the model is called to produce one image (10 is what the model was trained for).

**Status.** Written against the Fast-DDPM source; the pieces were tested on a CPU with a tiny model, but the notebook has **not yet been run on a Colab GPU or on Easley**. Report any error.

In [ ]:
# 1. Get the code and the pretrained checkpoint, and count the GPUs
import os, sys, glob, zipfile, argparse, subprocess, time, copy, io, re
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    CODE_DIR, CKPT = "Fast-DDPM", "ckpt_LDFDCT.pth"
    if not os.path.isdir(CODE_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/mirthAI/Fast-DDPM.git"], check=True)
    subprocess.run(["wget", "-q", "-c", "https://huggingface.co/SebastianJiang/FastDDPM/resolve/main/ckpt_LDFDCT.pth",
                    "-O", CKPT], check=True)
else:
    # Easley: paths from docs/hpc_quickstart.md. Point CKPT at <exp>/logs/<doc>/ckpt_<step>.pth for your own run.
    CODE_DIR = os.path.expanduser("~/ECE_551_problems_ML/fast-ddpm")
    CKPT = os.path.expanduser("~/carc-scratch/fastddpm-exp/logs/ldfdct_smoke_test/ckpt_1.pth")
    assert os.path.isdir(CODE_DIR) and os.path.isfile(CKPT), f"check CODE_DIR={CODE_DIR} and CKPT={CKPT}"

sys.path.insert(0, CODE_DIR)
import torch
N_GPU = torch.cuda.device_count()
if N_GPU == 0:
    raise RuntimeError("No GPU: in Colab use Runtime > Change runtime type > GPU; on Easley start the server on a GPU partition")
print("torch", torch.__version__, "|", "Colab" if IN_COLAB else "Easley", "|", N_GPU, "GPU(s)")
for i in range(N_GPU):
    print(f"  cuda:{i}  {torch.cuda.get_device_name(i)}  {torch.cuda.get_device_properties(i).total_memory / 2**30:.0f} GiB")

In [ ]:
# 2a. Build the model and load the pretrained weights (on the CPU; copies go to each GPU in the next cell)
import yaml
from models.diffusion import Model

def to_ns(d):
    ns = argparse.Namespace()
    for k, v in d.items():
        setattr(ns, k, to_ns(v) if isinstance(v, dict) else v)
    return ns

config = to_ns(yaml.safe_load(open(f"{CODE_DIR}/configs/ldfd_linear.yml")))
base_model = Model(config)
states = torch.load(CKPT, map_location="cpu", weights_only=False)
# checkpoint = [model, optimizer, epoch, step, EMA]; keys saved from DataParallel start with "module."
base_model.load_state_dict({k.replace("module.", "", 1): v for k, v in states[0].items()}, strict=True)
# The authors sample with the EMA (smoothed) weights, so copy them in, as EMAHelper.ema() does.
ema = states[-1]
n_params = sum(1 for _ in base_model.named_parameters())
n_ema = 0
for name, p in base_model.named_parameters():
    if name in ema:
        p.data.copy_(ema[name]); n_ema += 1
print(f"EMA weights applied to {n_ema}/{n_params} parameters; trained step {states[3]}")
base_model.eval()

In [ ]:
# 2b. One replica of the model per GPU
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import structural_similarity as ssim_fn
from concurrent.futures import ThreadPoolExecutor

DEVICES = [torch.device(f"cuda:{i}") for i in range(N_GPU)] or [torch.device("cpu")]
IMG = config.data.image_size
_replicas, _betas = {}, {}

def replica(dev):
    """The model and the noise schedule on one device. Built once, then reused (DataParallel rebuilds it on every call)."""
    if dev not in _replicas:
        _replicas[dev] = copy.deepcopy(base_model).to(dev).eval()
        _betas[dev] = torch.linspace(config.diffusion.beta_start, config.diffusion.beta_end,
                                     config.diffusion.num_diffusion_timesteps, dtype=torch.float64).float().to(dev)
    return _replicas[dev], _betas[dev]

def sync(dev):
    """GPU work is asynchronous: wait for it to finish before reading the clock."""
    if dev.type == "cuda":
        torch.cuda.synchronize(dev)

for d in DEVICES:
    replica(d)
print("replicas ready on", [str(d) for d in DEVICES])

In [ ]:
# 3. Images: your own pairs, the LDFDCT test set on Easley, or a synthetic test pattern
N_IMAGES = 32          # how many images to process in total (8 or more makes the sections below meaningful)

items = []             # each image: id (position in this list), name, ld (input, 0..1), fd (ground truth, 0..1 or None)

def add_item(name, ld, fd=None):
    f = lambda im: np.asarray(im.convert("L").resize((IMG, IMG)), dtype=np.float32) / 255.0
    items.append(dict(id=len(items), name=name, ld=f(ld), fd=None if fd is None else f(fd)))

def phantom(seed):
    """Overlapping ellipses plus noise. For timing only: the model was trained on CT, so scores on this mean little."""
    r = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:IMG, 0:IMG] / IMG
    fd = np.zeros((IMG, IMG), np.float32)
    for _ in range(6):
        cx, cy = r.uniform(.2, .8, 2); a, b = r.uniform(.05, .25, 2)
        fd[((xx - cx) / a) ** 2 + ((yy - cy) / b) ** 2 < 1] = r.uniform(.2, .9)
    ld = np.clip(fd + r.normal(0, .08, fd.shape), 0, 1).astype(np.float32)
    return Image.fromarray((ld * 255).astype(np.uint8)), Image.fromarray((fd * 255).astype(np.uint8))

INPUT_DIR = None
if IN_COLAB:
    from google.colab import files
    INPUT_DIR = "inputs"
    os.makedirs(INPUT_DIR, exist_ok=True)
    print("Upload NAME_ld.png / NAME_fd.png pairs, or a .zip of them (cancel to use the synthetic pattern)")
    for name, data in files.upload().items():
        open(f"{INPUT_DIR}/{name}", "wb").write(data)
        if name.lower().endswith(".zip"):
            zipfile.ZipFile(f"{INPUT_DIR}/{name}").extractall(INPUT_DIR)
else:
    INPUT_DIR = os.path.expanduser("~/carc-scratch/fastddpm-data/data/LD_FD_CT_test")

paths = sorted(glob.glob(f"{INPUT_DIR}/**/*_ld.png", recursive=True)) if os.path.isdir(INPUT_DIR) else []
if paths:
    paths = paths[::max(1, len(paths) // N_IMAGES)][:N_IMAGES]      # spread across the folder, not neighbouring slices
    for p in paths:
        fd = p.replace("_ld.png", "_fd.png")
        add_item(os.path.basename(p), Image.open(p), Image.open(fd) if os.path.exists(fd) else None)
    print(f"{len(items)} low-dose images from {INPUT_DIR}")
else:
    for k in range(N_IMAGES):
        ld, fd = phantom(k)
        add_item(f"phantom_{k:02d}", ld, fd)
    print(f"{len(items)} synthetic phantoms (no *_ld.png found): timings are real, scores are not meaningful")

In [ ]:
# 4. The sampler and the scoring, written so that every step stays on the device it was given
def make_seq(timesteps):
    """The time points the sampler visits: the repo's uniform scheduler (10 -> 0,99,...,999; 5 and 2 are subsets)."""
    skip = 1000 // timesteps
    seq = list(range(-1, 1000, skip)); seq[0] = 0
    return seq

def compute_alpha(beta, t):
    beta = torch.cat([torch.zeros(1, device=beta.device), beta])
    return (1 - beta).cumprod(dim=0).index_select(0, t + 1).view(-1, 1, 1, 1)

@torch.no_grad()
def ddim(x, x_img, seq, net, betas):
    """Deterministic DDIM (eta = 0), the same arithmetic as sg_generalized_steps in functions/denoising.py.
    The repo's version moves its tensors to 'cuda' (always GPU 0), so a replica on cuda:1 would break; this one
    keeps everything on x.device. With eta = 0 no fresh noise is added, so the starting noise decides the image."""
    n = x.size(0)
    seq_next = [-1] + list(seq[:-1])
    for i, j in zip(reversed(seq), reversed(seq_next)):
        t = torch.full((n,), float(i), device=x.device)
        t_next = torch.full((n,), float(j), device=x.device)
        at, at_next = compute_alpha(betas, t.long()), compute_alpha(betas, t_next.long())
        et = net(torch.cat([x_img, x], dim=1), t)                       # the model predicts the noise in x
        x0 = (x - et * (1 - at).sqrt()) / at.sqrt()                     # estimate of the clean image
        x = at_next.sqrt() * x0 + (1 - at_next).sqrt() * et             # jump to the next time point
    return x

SEED = 1244
def noise_for(ids):
    """Starting noise per image, drawn from a generator seeded with SEED + the image's id.
    So an image gets the same noise whichever GPU, batch or shard it lands in; this is what makes the
    merged-shards check in section 3 possible. (The repo draws one noise stream in order, so a sharded
    run with the repo's code would give each image different noise.)"""
    out = []
    for i in ids:
        g = torch.Generator().manual_seed(SEED + int(i))
        out.append(torch.randn(1, 1, IMG, IMG, generator=g))
    return torch.cat(out)

def run_batch(batch, dev, timesteps=10):
    """Denoise a list of images on one device. Returns an array (n, H, W) in 0..1."""
    net, betas = replica(dev)
    x_img = torch.from_numpy(np.stack([b["ld"] for b in batch]))[:, None].to(dev) * 2 - 1    # condition, in [-1, 1]
    x = noise_for([b["id"] for b in batch]).to(dev)
    out = ddim(x, x_img, make_seq(timesteps), net, betas)
    return ((out + 1) / 2).clamp(0, 1)[:, 0].float().cpu().numpy()

def score(pred, gt):
    """PSNR (dB) and SSIM of one image against its ground truth, on a 0..255 scale like the repo's evaluation."""
    mse = np.mean((pred * 255 - gt * 255) ** 2)
    psnr = float("inf") if mse == 0 else 20 * np.log10(255.0 / np.sqrt(mse))
    return psnr, ssim_fn(gt, pred, data_range=1.0)

run_batch(items[:1], DEVICES[0])        # warm-up call: the first call pays start-up costs that are not inference time
print("sampler ready; time points for 10 steps:", make_seq(10))

## 1. One GPU: how fast, and what do fewer steps cost?

Two measurements on `cuda:0`:
- **Batch size.** Throughput (images per second) as the batch grows. On the training side the H100 saturated at about batch 12; inference may behave differently.
- **Sampling steps.** Fewer model calls per image means less time. The model was trained on 10 steps, and 5 and 2 are subsets of the steps it saw, so they are a fair test of how much quality is lost.

The timer waits for the GPU to finish (`sync`) before reading the clock, otherwise it would only measure how fast Python launched the work.

In [ ]:
# 5. Batch size and sampling steps on one GPU
dev0 = DEVICES[0]
STEPS = 10

rows = []
for bs in [1, 2, 4, 8, 16]:
    if bs > len(items):
        break
    n_done = (len(items) // bs) * bs                         # only full batches, so every size does the same work
    try:
        run_batch(items[:bs], dev0, STEPS)                   # warm-up at this batch size (not timed)
        sync(dev0); t0 = time.perf_counter()
        for k in range(0, n_done, bs):
            run_batch(items[k:k + bs], dev0, STEPS)
        sync(dev0); dt = time.perf_counter() - t0
    except torch.cuda.OutOfMemoryError:
        print(f"batch {bs}: out of memory on {dev0}; stopping the sweep")
        torch.cuda.empty_cache()
        break
    rows.append(dict(batch=bs, images=n_done, seconds=round(dt, 2), images_per_s=round(n_done / dt, 2)))
sweep = pd.DataFrame(rows)
display(sweep)

rows = []
scored = [it for it in items if it["fd"] is not None][:16]
for ts in [10, 5, 2]:
    sync(dev0); t0 = time.perf_counter()
    outs = np.concatenate([run_batch(scored[k:k + 4], dev0, ts) for k in range(0, len(scored), 4)])
    sync(dev0); dt = time.perf_counter() - t0
    sc = [score(o, it["fd"]) for o, it in zip(outs, scored)]
    rows.append(dict(sampling_steps=ts, seconds=round(dt, 2), images_per_s=round(len(scored) / dt, 2),
                     PSNR=round(np.mean([s[0] for s in sc]), 2), SSIM=round(np.mean([s[1] for s in sc]), 4)))
steps_table = pd.DataFrame(rows)
display(steps_table)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(sweep.batch, sweep.images_per_s, "o-"); ax[0].set_xscale("log", base=2)
ax[0].set_xlabel("batch size"); ax[0].set_ylabel("images / s"); ax[0].set_title(f"throughput on {dev0}"); ax[0].set_ylim(bottom=0)
ax[1].bar([str(s) for s in steps_table.sampling_steps], steps_table.seconds)
ax[1].set_xlabel("sampling steps"); ax[1].set_ylabel(f"seconds for {len(scored)} images"); ax[1].set_title("time vs steps")
plt.tight_layout(); plt.show()

## 2. Splitting one batch across GPUs, by hand

This is what `torch.nn.DataParallel` does every time the model is called (it is already wrapped around the model in `sg_train` and `sg_sample`), written out in four steps:

1. **Scatter**: cut the batch into one chunk per GPU (a batch of 16 on 2 GPUs becomes 8 and 8; on 3 GPUs, 6, 5 and 5).
2. **Replicate**: put a copy of the model on each GPU. (DataParallel copies it again on every call; here the copies are made once.)
3. **Apply in parallel**: every GPU denoises its own chunk at the same time.
4. **Gather**: collect the chunks back, in order, into one result.

With one GPU the chunks are processed one after another, so the total should be close to the unsplit batch, not much shorter (it can differ a little, because the batch size changes the speed per image: see section 1). What the cell still proves is the bookkeeping: the gathered result must equal the unsplit result, up to rounding in the last digits.

In [ ]:
# 6. Scatter -> replicate -> apply in parallel -> gather
def split_batch(batch, devices, timesteps=10):
    chunks = np.array_split(np.arange(len(batch)), len(devices))                      # 1. scatter
    parallel = len(devices) > 1 and len(set(devices)) == len(devices)                 # distinct GPUs -> really at the same time
    def work(k):
        idx = chunks[k]
        if len(idx) == 0:
            return np.zeros((0, IMG, IMG), np.float32), 0.0
        d = devices[k]                                                                # 2. the replica for this device
        sync(d); t0 = time.perf_counter()
        out = run_batch([batch[i] for i in idx], d, timesteps)                        # 3. apply
        sync(d)
        return out, time.perf_counter() - t0
    t0 = time.perf_counter()
    if parallel:
        with ThreadPoolExecutor(len(devices)) as ex:
            res = list(ex.map(work, range(len(devices))))
    else:
        res = [work(k) for k in range(len(devices))]
    wall = time.perf_counter() - t0
    return np.concatenate([r[0] for r in res]), [r[1] for r in res], [len(c) for c in chunks], wall   # 4. gather

BATCH = min(16, len(items))
batch = items[:BATCH]
N_SPLIT = max(2, N_GPU)                                    # split into 2 chunks even on one GPU, to show the bookkeeping
devices = DEVICES if N_GPU >= 2 else [DEVICES[0]] * N_SPLIT
def median_run(devs, reps=3):
    split_batch(batch, devs)                                 # warm-up: the first call at a new batch shape is slower
    runs = [split_batch(batch, devs) for _ in range(reps)]
    return sorted(runs, key=lambda r: r[3])[reps // 2]       # the run with the median wall-clock time

whole, _, _, wall_whole = median_run([DEVICES[0]])
split, chunk_s, sizes, wall_split = median_run(devices)

print(f"batch of {BATCH} -> chunks of {sizes} on {[str(d) for d in devices]}")
print(f"  one device, whole batch : {wall_whole:6.2f} s")
print(f"  split into {len(devices)} chunks   : {wall_split:6.2f} s   (chunk times {', '.join(f'{c:.2f}' for c in chunk_s)} s)"
      + ("" if len(set(devices)) > 1 else "   <- one GPU: the chunks ran one after another"))
print(f"  gathered result vs whole batch: max pixel difference {np.abs(split - whole).max():.2e} (on a 0..1 scale)")
if len(set(devices)) > 1:
    print(f"  speed-up {wall_whole / wall_split:.2f}x, efficiency {wall_whole / wall_split / len(devices):.2f}")

## 3. Sharding the test set: no communication at all

For inference the GPUs do not need to talk to each other. Give GPU *k* the images whose id leaves remainder *k* when divided by N, let each one work alone, then put the per-image results back together by id. On the cluster this is a **job array** (one task per shard) or one **`torchrun`** launch (one process per shard); here the same logic runs inside the notebook.

Two things have to hold, and the cell below checks both:
- **The merge must be by image, not by shard average.** Averages of averages are wrong when shards have different sizes.
- **The merged result must equal the one-GPU result.** That needs each image to get the same starting noise wherever it runs, which is why `noise_for` seeds the noise from the image's id. (Two runs can still differ in the last digits: the GPU may pick a different algorithm for a different batch size.)

The tolerance used below, 0.01 on a 0..1 scale (about 2.5 grey levels), and 0.01 dB are thresholds we chose, not standards.

In [ ]:
# 7. Sharded inference, merged by image id, checked against the one-GPU run
def run_shard(k, n_shards, devices, timesteps, batch_size):
    dev = devices[k % len(devices)]
    mine = [it for it in items if it["id"] % n_shards == k]                  # the shard: ids k, k+N, k+2N, ...
    rows, outs = [], {}
    sync(dev); t0 = time.perf_counter()
    for s in range(0, len(mine), batch_size):
        b = mine[s:s + batch_size]
        for it, o in zip(b, run_batch(b, dev, timesteps)):
            outs[it["id"]] = o
            p, q = score(o, it["fd"]) if it["fd"] is not None else (np.nan, np.nan)
            rows.append(dict(id=it["id"], name=it["name"], shard=k, psnr=p, ssim=q))
    sync(dev)
    return rows, outs, time.perf_counter() - t0

def sharded(n_shards, devices, timesteps=10, batch_size=4):
    parallel = len(devices) >= n_shards > 1 and len(set(devices[:n_shards])) == n_shards
    t0 = time.perf_counter()
    if parallel:
        with ThreadPoolExecutor(n_shards) as ex:
            res = list(ex.map(lambda k: run_shard(k, n_shards, devices, timesteps, batch_size), range(n_shards)))
    else:
        res = [run_shard(k, n_shards, devices, timesteps, batch_size) for k in range(n_shards)]
    wall = time.perf_counter() - t0
    table = pd.DataFrame([r for rr, _, _ in res for r in rr]).sort_values("id").reset_index(drop=True)   # merge by id
    outs = {i: o for _, oo, _ in res for i, o in oo.items()}
    return table, outs, [t for _, _, t in res], wall, parallel

ref_table, ref_outs, _, ref_wall, _ = sharded(1, DEVICES)      # also a warm-up for the batch shapes used below
summary, results = [], {}
SHARD_COUNTS = sorted({1, 2, 4, max(N_GPU, 1)})            # includes one shard per GPU, so every GPU gets work
for n in SHARD_COUNTS:
    table, outs, shard_s, wall, parallel = sharded(n, DEVICES)
    results[n] = (table, outs, shard_s)
    diff = max(np.abs(outs[i] - ref_outs[i]).max() for i in ref_outs)
    summary.append(dict(shards=n, images=len(table), ran="at the same time" if parallel else "one after another",
                        wall_s=round(wall, 2), slowest_shard_s=round(max(shard_s), 2),
                        PSNR_mean=round(table.psnr.mean(), 3), SSIM_mean=round(table.ssim.mean(), 4),
                        max_pixel_diff_vs_1_shard=f"{diff:.1e}",
                        same_as_1_shard=bool(diff < 0.01 and abs(table.psnr.mean() - ref_table.psnr.mean()) < 0.01
                                             if table.psnr.notna().all() else diff < 0.01)))
display(pd.DataFrame(summary))
print("merged by image id; 'ran' says whether the shards really overlapped in time (needs one GPU per shard).")
if N_GPU < 2:
    print("One GPU: the shards ran one after another, so wall_s does not shrink. The correctness columns are the point here.")

In [ ]:
# 8. Per-shard time: was the work balanced?
n = N_GPU if N_GPU > 1 else max(results)                  # one shard per GPU when there are several GPUs
table, outs, shard_s = results[n]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].bar([f"shard {k}" for k in range(len(shard_s))], shard_s)
ax[0].set_ylabel("seconds"); ax[0].set_title(f"time per shard ({n} shards)")
ax[1].hist(table.psnr.dropna(), bins=20); ax[1].set_xlabel("PSNR per image (dB)"); ax[1].set_ylabel("images")
ax[1].set_title("scores of the merged result")
plt.tight_layout(); plt.show()
display(table.head(8))

## 4. Look at the merged result

In [ ]:
# 9. input | generated | ground truth | error, for the merged result of the shard run above
def show(rows):
    fig, axes = plt.subplots(len(rows), 4, figsize=(13, 3.3 * len(rows)), squeeze=False)
    for r, (title, inp, out, gt) in enumerate(rows):
        panels = [("input (low dose)", inp), ("generated", out), ("ground truth", gt),
                  ("|generated - truth|", None if gt is None else np.abs(out - gt))]
        for c, (name, img) in enumerate(panels):
            ax = axes[r][c]; ax.axis("off")
            if img is not None:
                ax.imshow(img, cmap="hot" if c == 3 else "gray", vmin=0, vmax=0.3 if c == 3 else 1)
                ax.set_title(name, fontsize=10)
        if gt is not None:
            p, q = score(out, gt)
            axes[r][0].text(0, -0.04, f"{title}   PSNR {p:.2f} dB  SSIM {q:.4f}", transform=axes[r][0].transAxes, fontsize=9, va="top")
    plt.tight_layout(); plt.show()

N_SHOW = 4
pick = [items[i] for i in np.linspace(0, len(items) - 1, N_SHOW).round().astype(int)]
show([(it["name"], it["ld"], outs[it["id"]], it["fd"]) for it in pick])

## 5. Timings measured on Easley

The cluster scripts write one result line per run. Upload the files here to see the speed-up. Today that means the **training** benchmark: run `slurm/scaling_bench.sh` once on 1 H100 and once on 2 H100, and each run leaves a `scale_result_<GPU>_<N>gpu_bs<B>.txt` in the folder you submitted from. Copy them to your computer:

```bash
scp "lc2025@easley.alliance.unm.edu:~/ECE_551_problems_ML/slurm/scale_result_*.txt" .
```

The same cell will read the inference benchmark's result lines later, as long as they use the same `key=value` format.

How to read the charts:
- **Speed-up** = (seconds per step on 1 GPU) / (seconds per step on N GPUs), measured at the same global batch and GPU model.
- **The dashed line** is perfect scaling (N times faster). **The dotted line** is the most two H100 can reach at a global batch of 16 (1.88x): one H100 takes 0.182 s per step at batch 8 and 0.342 s at batch 16, so splitting a batch of 16 cannot be twice as fast. This is computed from two measurements, not a separate one.
- **Efficiency** = speed-up / N.

In [ ]:
# 10. Plot scale_result_*.txt files from the cluster
H100_BOUND_2GPU = 0.342 / 0.182      # computed: batch-16 step time / batch-8 step time on one H100

RESULTS_DIR = "."
if IN_COLAB:
    from google.colab import files
    os.makedirs("cluster_results", exist_ok=True)
    for name, data in files.upload().items():
        open(f"cluster_results/{name}", "wb").write(data)
    RESULTS_DIR = "cluster_results"
else:
    RESULTS_DIR = os.path.expanduser("~/ECE_551_problems_ML/slurm")

recs = []
for p in sorted(glob.glob(f"{RESULTS_DIR}/scale_result_*.txt")):
    for line in open(p):
        kv = dict(x.split("=", 1) for x in line.split() if "=" in x)
        if "tag" in kv:
            recs.append(kv)
if not recs:
    print(f"no scale_result_*.txt in {RESULTS_DIR}: run slurm/scaling_bench.sh on the cluster first")
else:
    num = lambda v: float(v) if v not in (None, "n/a") else np.nan
    df = pd.DataFrame([dict(tag=r["tag"], gpu_model=r["tag"].split("_")[0], gpus=int(r["gpus"]), global_batch=int(r["global_batch"]),
                            status=r.get("status", "?"), s_per_step=num(r.get("median_s_per_step")),
                            images_per_s=num(r.get("images_per_s")), peak_mem_MiB=r.get("peak_mem_MiB_per_gpu", ""),
                            util_pct=r.get("median_util_pct_per_gpu", "")) for r in recs])
    base = df[(df.gpus == 1) & (df.status == "OK")].set_index(["gpu_model", "global_batch"]).s_per_step
    df["speedup"] = [base.get((g, b), np.nan) / s if st == "OK" else np.nan
                     for g, b, s, st in zip(df.gpu_model, df.global_batch, df.s_per_step, df.status)]
    df["efficiency"] = df.speedup / df.gpus
    display(df.sort_values(["gpu_model", "global_batch", "gpus"]).round(3))

    ok = df[df.status == "OK"].sort_values("gpus")
    if len(ok):
        fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
        labels = [f"{r.gpu_model}\n{r.gpus} GPU, batch {r.global_batch}" for r in ok.itertuples()]
        ax[0].bar(labels, ok.s_per_step); ax[0].set_ylabel("seconds per step"); ax[0].set_title("time per training step")
        ax[1].bar(labels, ok.images_per_s); ax[1].set_ylabel("images / s"); ax[1].set_title("throughput")
        sp = ok.dropna(subset=["speedup"])
        ax[2].plot(sp.gpus, sp.speedup, "o-", label="measured")
        ax[2].plot([1, sp.gpus.max()], [1, sp.gpus.max()], "--", color="gray", label="perfect scaling")
        if (sp.gpu_model == "H100").any() and (sp.global_batch == 16).any():
            ax[2].axhline(H100_BOUND_2GPU, ls=":", color="k", label=f"H100 bound at batch 16 ({H100_BOUND_2GPU:.2f}x)")
        ax[2].set_xlabel("GPUs"); ax[2].set_ylabel("speed-up"); ax[2].set_title("speed-up over 1 GPU"); ax[2].legend(fontsize=8)
        for a in ax[:2]:
            a.tick_params(axis="x", labelsize=7)
        plt.tight_layout(); plt.show()

## How this maps to the cluster

| In this notebook | On Easley |
|---|---|
| `run_batch` on one device | one GPU, one process: `python fast_ddpm_main.py --sample ...` |
| Section 2, a batch split across GPUs | the same command given 2 GPUs: `sg_sample` already wraps the model in `DataParallel`, so no code change |
| Section 3, shard *k* of N | a job array (`--array=0-3`, `SLURM_ARRAY_TASK_ID` is *k*) or `torchrun --nproc_per_node=N` (`LOCAL_RANK` is *k*), plus a patch so each shard writes a per-image table |
| Merge by image id | one small script that concatenates the per-shard tables and averages over rows |
| `noise_for` | a per-image seed in the patched sampler |

**What this notebook does not show:** gradient synchronization (inference has none; training uses DDP, see `docs/multi_gpu_background.md`), more than the GPUs your runtime has, or any speed-up on one GPU.